In [24]:
import numpy as np
import pandas as pd
from pycaret.classification import *

In [25]:
np.random.seed(42)
rows = 50

# Create sample data with 10 columns.
data = {
    "num1": np.random.randint(0, 100, size=rows),
    "num2": np.random.randn(rows) * 10,
    "cat1": np.random.choice(["apple", "banana", "cherry"], size=rows),
    "cat2": np.random.choice(["red", "green", "blue"], size=rows),
    "num3": np.random.uniform(0, 50, size=rows),
    "num4": np.random.randint(0, 200, size=rows),
    "text": [f"Sample {i}" for i in range(rows)],
    "bool": np.random.choice([True, False], size=rows),
    "num5": np.random.randn(rows) * 5,
    "cat3": np.random.choice(["X", "Y", "Z"], size=rows)
}
df = pd.DataFrame(data)

for col in ["num2", "num3", "text"]:
    missing_indices = np.random.choice(df.index, size=5, replace=False)
    df.loc[missing_indices, col] = np.nan

df.loc[10] = df.loc[0]
df.loc[20] = df.loc[0]

data = df.copy()
data

,num1,num2,cat1,cat2,num3,num4,text,bool,num5,cat3
0,51,-20.267196,cherry,red,33.606777,98,Sample 0,True,0.291044,Y
1,92,11.194236,banana,red,NaN,152,Sample 1,False,-5.714851,Y
2,14,7.791926,cherry,red,11.881877,92,Sample 2,True,1.788937,X
3,71,-11.010978,apple,blue,36.410817,145,Sample 3,False,2.803923,X
4,60,11.302282,apple,green,18.389157,127,Sample 4,True,5.415256,X
5,20,NaN,banana,red,NaN,109,Sample 5,False,5.269010,X
6,82,-3.864730,cherry,red,31.676486,81,Sample 6,False,-6.888347,X
7,86,-11.587702,cherry,red,26.788734,193,Sample 7,False,-4.689125,Z
8,74,5.661128,banana,blue,4.514489,53,Sample 8,True,2.575176,Z
9,74,-7.044535,cherry,blue,41.765125,162,Sample 9,False,2.568930,X


In [26]:
exp = setup(
    data=data, 
    target='cat3', 
    session_id=42, 
    verbose=False,
    imputation_type='simple',      # 使用简单的缺失值填充方式
    numeric_imputation='mean',       # 数值型数据用均值填充
    categorical_imputation='mode'    # 类别型数据用众数填充
)
    
# 获取预处理后的训练数据
X = get_config('X_train')
y = get_config('y_train')
    
# 合并特征和目标列，形成预处理后的完整数据集
processed_data = X.copy()
processed_data['cat3'] = y
processed_data

,num1,num2,cat1,cat2,num3,num4,text,bool,num5,cat3
18,29,-9.071866,cherry,blue,32.258640,147,Sample 18,True,-3.864126,X
24,32,4.608163,cherry,red,17.053318,127,NaN,True,3.431301,Y
17,87,-1.716288,cherry,green,11.324789,47,Sample 17,True,3.794846,Y
2,14,7.791926,cherry,red,11.881877,92,Sample 2,True,1.788937,X
22,59,2.634861,banana,red,46.836498,194,Sample 22,True,11.573293,Z
37,61,1.736021,apple,green,31.655073,26,Sample 37,False,3.169595,Y
30,90,-0.435368,apple,red,40.861111,21,Sample 30,True,-3.576519,X
13,2,0.666573,cherry,green,29.544647,67,Sample 13,False,5.677828,Z
19,37,11.886257,cherry,blue,8.718322,127,NaN,False,-1.184093,X
25,75,NaN,cherry,green,5.673676,32,Sample 25,False,-8.063580,Y


In [27]:
# Remove duplicate rows
data = data.drop_duplicates()
# Remove Leading/Trailing Spaces
data.columns = data.columns.str.strip()

In [28]:
# Define maximum allowed categories (less than 1% of rows)
max_categories_threshold = int(len(data) * 0.01)

# Identify categorical columns (object or category dtypes) that meet the threshold
eligible_cat_features = [
    col for col in data.select_dtypes(include=['object', 'category', 'bool']).columns
    if data[col].nunique() <= max_categories_threshold]

print("Eligible categorical features for encoding:", eligible_cat_features)

# Encode each eligible categorical column using pd.factorize
for col in eligible_cat_features:
    data[col], _ = pd.factorize(data[col])


Eligible categorical features for encoding: []


In [29]:
target = 'null'
# Transit target from front end

# Detect target column if not provided
if target == 'null' or target not in data.columns:
    def detect_target_column(df):
        possible_target = None
        for col in df.columns:
            unique_values = df[col].nunique()
            # Check if the column has no missing values and is suitable for classification
            if unique_values == 2 and df[col].isnull().sum() == 0:  # Binary classification without missing values
                possible_target = col
            elif 2 < unique_values < len(data) * 0.01 and df[col].isnull().sum() == 0:  # Multi-class classification without missing values
                possible_target = col
        return possible_target # Target column is more possible on the right side
    
    target = detect_target_column(data)

# find if target is found
if target:
    print(f"Detected target column: {target}")
else:
    print("No suitable target column detected. Please specify it manually.")

if target in eligible_cat_features:
    eligible_cat_features.remove(target)

Detected target column: bool


In [30]:
import re

def detect_ignore_features(df, keywords=None):
    if keywords is None:
        # Only ignore columns that are very likely identifiers or irrelevant
        keywords = ['row', 'id', 'name']
        
    ignore_features = []
    
    for col in df.columns:
        # Check if column name contains any of the keywords
        if any(re.search(keyword, col, flags=re.IGNORECASE) for keyword in keywords):
            ignore_features.append(col)
    
    return ignore_features

# Usage with your DataFrame
ignore_features = detect_ignore_features(data)
print("Automatically detected ignore features:", ignore_features)


Automatically detected ignore features: []


In [31]:
clf = setup(
    data=data,
    target=target,
    
    # Data cleaning configuration
    numeric_imputation="median",  # Fill missing numerical values with the median
    categorical_imputation='mode',  # Fill missing categorical values with the mode
    remove_multicollinearity=True,  # Remove multicollinearity features
    multicollinearity_threshold=1.0, 
    remove_outliers=True,  # Automatically handle outliers using the IQR method
    outliers_threshold=0.05,  # Outlier detection threshold
    fold_strategy='stratifiedkfold',  # Use stratified K-fold cross-validation
    categorical_features=eligible_cat_features,
    normalize=True,

    # Categorical feature handling
    ignore_features=ignore_features,  # Ignore irrelevant features
    
    # Other automation settings
    fix_imbalance=True,  # Automatically handle class imbalance
    session_id=42  # Random seed
)


,Description,Value
0,Session id,42
1,Target,bool
2,Target type,Binary
3,Original data shape,"(48, 10)"
4,Transformed data shape,"(49, 16)"
5,Transformed train set shape,"(34, 16)"
6,Transformed test set shape,"(15, 16)"
7,Numeric features,5
8,Categorical features,4
9,Rows with missing values,27.1%


In [32]:
# Get the cleaned data
cleaned_data = get_config('X_train')  # Retrieve the cleaned data (excluding the target column)
cleaned_data[target] = get_config('y_train')  # Re-add the target column

cleaned_data

,num1,num2,cat1,cat2,num3,num4,text,num5,cat3,bool
41,50,11.647686,banana,red,NaN,96,Sample 41,4.262167,Y,True
17,87,-1.716288,cherry,green,11.324789,47,Sample 17,3.794846,Y,True
33,91,-7.956310,cherry,red,12.092614,108,Sample 33,1.082293,Y,True
39,46,2.411222,apple,blue,17.460478,20,Sample 39,0.932272,Y,False
25,75,NaN,cherry,green,5.673676,32,Sample 25,-8.063580,Y,False
5,20,NaN,banana,red,NaN,109,Sample 5,5.269010,X,False
9,74,-7.044535,cherry,blue,41.765125,162,Sample 9,2.568930,X,False
8,74,5.661128,banana,blue,4.514489,53,Sample 8,2.575176,Z,True
19,37,11.886257,cherry,blue,8.718322,127,NaN,-1.184093,X,False
35,79,6.777674,banana,red,44.860786,181,Sample 35,-3.258002,X,False


In [33]:
# # Retrieve and compare all models
# compare_models(sort='AUC')

# # Get the comparison results
# model_comparison = pull()
# print(model_comparison)